# BGRU Baseline Model - Step-by-Step Implementation

This notebook provides a step-by-step walkthrough of building the Bidirectional GRU (BGRU) model for BankNifty directional prediction. It's designed to help with debugging and understanding each component.

## Table of Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Load and Prepare Data](#2-load-and-prepare-data)
3. [Data Normalization](#3-data-normalization)
4. [Sequence Preparation](#4-sequence-preparation)
5. [Build BGRU Model](#5-build-bgru-model)
6. [Training Setup](#6-training-setup)
7. [Training Loop](#7-training-loop)
8. [Evaluation](#8-evaluation)
9. [Visualization](#9-visualization)
10. [Save and Load Model](#10-save-and-load-model)

## 1. Setup and Imports

In [ ]:
# Standard library imports
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

# Data manipulation
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bar
from tqdm.notebook import tqdm

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load and Prepare Data

In [ ]:
# Define paths
DATA_DIR = Path('../data/processed/')
CHECKPOINT_DIR = Path('../models/checkpoints/')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Load data or create sample
if (DATA_DIR / 'train.csv').exists():
    print("Loading processed data...")
    train_df = pd.read_csv(DATA_DIR / 'train.csv', index_col=0, parse_dates=True)
    val_df = pd.read_csv(DATA_DIR / 'val.csv', index_col=0, parse_dates=True)
    test_df = pd.read_csv(DATA_DIR / 'test.csv', index_col=0, parse_dates=True)
else:
    print("Creating sample data for demonstration...")
    np.random.seed(42)
    
    def create_sample_df(n_samples):
        dates = pd.date_range(start='2024-01-01', periods=n_samples, freq='15min')
        base_price = 45000
        close = base_price + np.cumsum(np.random.randn(n_samples) * 50)
        high = close + np.abs(np.random.randn(n_samples) * 30)
        low = close - np.abs(np.random.randn(n_samples) * 30)
        open_price = close + np.random.randn(n_samples) * 20
        volume = np.random.randint(10000, 100000, n_samples).astype(float)
        
        df = pd.DataFrame({
            'open': open_price,
            'high': high,
            'low': low,
            'close': close,
            'volume': volume
        }, index=dates)
        
        df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
        df = df.iloc[:-1]
        return df
    
    train_df = create_sample_df(2000)
    val_df = create_sample_df(500)
    test_df = create_sample_df(500)

print(f"Train: {len(train_df):,} samples")
print(f"Val: {len(val_df):,} samples")
print(f"Test: {len(test_df):,} samples")

In [ ]:
# Examine data structure
print("Training Data Sample:")
display(train_df.head())
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nTarget Distribution (Train):")
print(train_df['target'].value_counts())

## 3. Data Normalization

We use rolling z-score normalization to standardize the OHLCV features.

In [ ]:
# Define constants
MIN_STD = 1e-8  # Minimum std to avoid division by zero
OHLCV_COLS = ['open', 'high', 'low', 'close', 'volume']

def normalize_data(df, method='rolling_zscore', window=100):
    """
    Normalize OHLCV data using rolling z-score.
    
    Args:
        df: DataFrame with OHLCV columns
        method: Normalization method
        window: Rolling window size
    
    Returns:
        Normalized DataFrame
    """
    df = df.copy()
    
    for col in OHLCV_COLS:
        if col in df.columns:
            rolling_mean = df[col].rolling(window=window, min_periods=1).mean()
            rolling_std = df[col].rolling(window=window, min_periods=1).std()
            rolling_std = rolling_std.replace(0, MIN_STD)
            df[col] = (df[col] - rolling_mean) / rolling_std
    
    # Handle NaN values
    df = df.ffill().bfill()
    
    return df

# Test normalization on a sample
sample_normalized = normalize_data(train_df.head(200))
print("Normalized Data Sample:")
display(sample_normalized.head())
print(f"\nNormalized Stats:")
display(sample_normalized[OHLCV_COLS].describe())

In [ ]:
# Visualize normalization effect
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Original close prices
axes[0, 0].plot(train_df['close'].head(500).values)
axes[0, 0].set_title('Original Close Prices')
axes[0, 0].set_xlabel('Time Index')
axes[0, 0].set_ylabel('Price')

# Normalized close prices
normalized_train = normalize_data(train_df.head(500))
axes[0, 1].plot(normalized_train['close'].values)
axes[0, 1].set_title('Normalized Close Prices (Rolling Z-Score)')
axes[0, 1].set_xlabel('Time Index')
axes[0, 1].set_ylabel('Normalized Value')
axes[0, 1].axhline(0, color='r', linestyle='--', alpha=0.5)

# Original distribution
axes[1, 0].hist(train_df['close'].head(500), bins=50, edgecolor='white')
axes[1, 0].set_title('Original Close Distribution')
axes[1, 0].set_xlabel('Price')

# Normalized distribution
axes[1, 1].hist(normalized_train['close'], bins=50, edgecolor='white')
axes[1, 1].set_title('Normalized Close Distribution')
axes[1, 1].set_xlabel('Normalized Value')

plt.tight_layout()
plt.show()

## 4. Sequence Preparation

Create sliding window sequences for the BGRU model. Each sequence contains 60 time steps (15 hours of 15-min data).

In [ ]:
def prepare_sequences(df, sequence_length=60, normalize=True):
    """
    Prepare sequences for BGRU model.
    
    Args:
        df: DataFrame with OHLCV and target columns
        sequence_length: Number of time steps per sequence
        normalize: Whether to apply normalization
    
    Returns:
        X: Sequences array [num_samples, sequence_length, 5]
        y: Target array [num_samples]
    """
    # Normalize if requested
    if normalize:
        df_norm = normalize_data(df, method='rolling_zscore', window=100)
    else:
        df_norm = df.copy()
    
    # Extract features
    features = df_norm[OHLCV_COLS].values
    
    # Extract targets
    targets = df['target'].values if 'target' in df.columns else None
    
    # Create sequences
    X, y = [], []
    for i in range(len(features) - sequence_length):
        X.append(features[i:i + sequence_length])
        if targets is not None:
            y.append(targets[i + sequence_length - 1])
    
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32) if targets is not None else np.array([])
    
    return X, y

# Test sequence preparation
SEQUENCE_LENGTH = 60

print(f"Preparing sequences with length {SEQUENCE_LENGTH}...")
X_train, y_train = prepare_sequences(train_df, sequence_length=SEQUENCE_LENGTH)
X_val, y_val = prepare_sequences(val_df, sequence_length=SEQUENCE_LENGTH)
X_test, y_test = prepare_sequences(test_df, sequence_length=SEQUENCE_LENGTH)

print(f"\nSequence Shapes:")
print(f"  X_train: {X_train.shape} -> [samples, sequence_length, features]")
print(f"  y_train: {y_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  y_val: {y_val.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")

In [ ]:
# Visualize a sample sequence
sample_idx = 0
sample_sequence = X_train[sample_idx]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(OHLCV_COLS):
    ax = axes[i]
    ax.plot(sample_sequence[:, i], marker='o', markersize=2)
    ax.set_title(f'{col.upper()} - Sample Sequence')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Normalized Value')
    ax.grid(True, alpha=0.3)

# Add info in last subplot
axes[-1].text(0.5, 0.5, f'Sample Sequence #{sample_idx}\n\n'
              f'Shape: {sample_sequence.shape}\n'
              f'Target: {y_train[sample_idx]}\n'
              f'(0=DOWN, 1=UP)',
              ha='center', va='center', fontsize=14,
              transform=axes[-1].transAxes)
axes[-1].axis('off')

plt.suptitle('Sample Input Sequence Visualization', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Build BGRU Model

Define the Bidirectional GRU architecture step by step.

In [ ]:
class BGRUModel(nn.Module):
    """
    Bidirectional GRU Model for Binary Classification.
    
    Architecture:
        Input: [batch, sequence_length, 5]
        BGRU Layer 1: 128 units bidirectional (256 output)
        Dropout: 0.3
        BGRU Layer 2: 64 units bidirectional (128 output)
        Dropout: 0.3
        Dense: 32 units, ReLU
        Dropout: 0.2
        Output: 1 unit, Sigmoid
    """
    
    def __init__(self, input_dim=5, hidden_dim=128, num_layers=2, dropout=0.3):
        super(BGRUModel, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        
        # Layer 1: BGRU with 128 hidden units
        # Output: [batch, seq_len, 256] (128 forward + 128 backward)
        self.gru1 = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,  # 128
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0
        )
        self.dropout1 = nn.Dropout(dropout)
        
        # Layer 2: BGRU with 64 hidden units
        # Input: 256 (from layer 1), Output: [batch, seq_len, 128]
        self.gru2 = nn.GRU(
            input_size=hidden_dim * 2,  # 256
            hidden_size=hidden_dim // 2,  # 64
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0
        )
        self.dropout2 = nn.Dropout(dropout)
        
        # Dense layer: 128 -> 32 with ReLU
        self.fc1 = nn.Linear(hidden_dim, 32)  # 128 -> 32
        self.relu = nn.ReLU()
        self.dropout3 = nn.Dropout(0.2)
        
        # Output layer: 32 -> 1 with Sigmoid
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor [batch, sequence_length, input_dim]
        
        Returns:
            Output tensor [batch, 1]
        """
        # BGRU Layer 1
        out, _ = self.gru1(x)  # [batch, seq_len, 256]
        out = self.dropout1(out)
        
        # BGRU Layer 2
        out, _ = self.gru2(out)  # [batch, seq_len, 128]
        out = self.dropout2(out)
        
        # Take last time step output
        out = out[:, -1, :]  # [batch, 128]
        
        # Dense layers
        out = self.fc1(out)  # [batch, 32]
        out = self.relu(out)
        out = self.dropout3(out)
        
        # Output
        out = self.fc2(out)  # [batch, 1]
        out = self.sigmoid(out)
        
        return out

# Create model
model = BGRUModel(
    input_dim=5,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3
).to(device)

print("Model Architecture:")
print(model)

In [ ]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Parameter breakdown by layer
print(f"\nParameter Breakdown:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.numel():,} params")

In [ ]:
# Test forward pass
print("Testing forward pass...")
sample_batch = torch.FloatTensor(X_train[:8]).to(device)
print(f"  Input shape: {sample_batch.shape}")

with torch.no_grad():
    output = model(sample_batch)
print(f"  Output shape: {output.shape}")
print(f"  Output range: [{output.min().item():.4f}, {output.max().item():.4f}]")
print(f"  Sample outputs: {output.flatten()[:4].tolist()}")
print("\nForward pass successful!")

## 6. Training Setup

Configure the training components: data loaders, loss function, optimizer, and scheduler.

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 20  # Reduced for notebook demo
PATIENCE = 10
GRADIENT_CLIP = 1.0

print("Training Configuration:")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Epochs: {EPOCHS}")
print(f"  Early Stopping Patience: {PATIENCE}")
print(f"  Gradient Clipping: {GRADIENT_CLIP}")

In [ ]:
# Create data loaders
print("Creating data loaders...")

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)
y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

In [ ]:
# Loss function
criterion = nn.BCELoss()
print(f"Loss Function: {criterion}")

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
print(f"Optimizer: {optimizer}")

# Learning rate scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)
print(f"Scheduler: ReduceLROnPlateau(factor=0.5, patience=5)")

## 7. Training Loop

Train the model with early stopping and learning rate scheduling.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, gradient_clip):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in loader:
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip)
        
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item() * batch_X.size(0)
        predictions = (outputs >= 0.5).float()
        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = total_loss / total
    accuracy = correct / total
    
    return avg_loss, accuracy


def validate(model, loader, criterion):
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item() * batch_X.size(0)
            predictions = (outputs >= 0.5).float()
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
    
    avg_loss = total_loss / total
    accuracy = correct / total
    
    return avg_loss, accuracy

In [ ]:
# Training loop
print("Starting training...")
print("=" * 60)

# History tracking
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

# Early stopping
best_val_acc = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, GRADIENT_CLIP
    )
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion)
    
    # Update scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Print progress
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.6f}")
    
    # Early stopping check
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Save best model
        best_model_state = model.state_dict().copy()
        print(f"  -> New best model! Val Acc: {val_acc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

print("=" * 60)
print(f"Training complete! Best Val Acc: {best_val_acc:.4f}")

# Load best model
model.load_state_dict(best_model_state)

## 8. Evaluation

Evaluate the model on the test set.

In [ ]:
# Prepare test data
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1).to(device)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Evaluate on test set
test_loss, test_acc = validate(model, test_loader, criterion)

print("Test Set Evaluation:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")

In [ ]:
# Generate predictions
model.eval()
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for batch_X, _ in test_loader:
        outputs = model(batch_X)
        all_probabilities.extend(outputs.cpu().numpy())
        predictions = (outputs >= 0.5).float()
        all_predictions.extend(predictions.cpu().numpy())

predictions = np.array(all_predictions).flatten()
probabilities = np.array(all_probabilities).flatten()

print(f"Predictions shape: {predictions.shape}")
print(f"Prediction distribution:")
print(f"  DOWN (0): {(predictions == 0).sum()}")
print(f"  UP (1): {(predictions == 1).sum()}")

In [ ]:
# Classification report
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(y_test, predictions, target_names=['DOWN', 'UP']))

# Confusion matrix
cm = confusion_matrix(y_test, predictions)
print("\nConfusion Matrix:")
print(cm)

## 9. Visualization

Visualize training curves and results.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(epochs_range, history['train_acc'], 'b-', label='Train Acc')
axes[0, 1].plot(epochs_range, history['val_acc'], 'r-', label='Val Acc')
axes[0, 1].set_title('Training and Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Learning rate
axes[1, 0].plot(epochs_range, history['lr'], 'g-')
axes[1, 0].set_title('Learning Rate Schedule')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].grid(True, alpha=0.3)

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'],
            ax=axes[1, 1])
axes[1, 1].set_title('Confusion Matrix')
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'training_curves.png', dpi=150)
plt.show()

print(f"Training curves saved to {CHECKPOINT_DIR / 'training_curves.png'}")

In [ ]:
# Probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall probability distribution
axes[0].hist(probabilities, bins=50, edgecolor='white', alpha=0.7)
axes[0].axvline(0.5, color='red', linestyle='--', label='Threshold')
axes[0].set_title('Prediction Probability Distribution')
axes[0].set_xlabel('Probability')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Probability by actual class
for label, color, name in [(0, '#ff6b6b', 'DOWN'), (1, '#4ecdc4', 'UP')]:
    mask = y_test == label
    axes[1].hist(probabilities[mask], bins=30, alpha=0.5, 
                 color=color, label=f'Actual {name}')

axes[1].axvline(0.5, color='red', linestyle='--', label='Threshold')
axes[1].set_title('Probability Distribution by Actual Class')
axes[1].set_xlabel('Predicted Probability of UP')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Save and Load Model

Demonstrate model persistence.

In [ ]:
# Save model checkpoint
checkpoint_path = CHECKPOINT_DIR / 'bgru_baseline.pt'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'input_dim': 5,
    'hidden_dim': 128,
    'num_layers': 2,
    'dropout': 0.3,
    'training_history': history,
    'best_val_acc': best_val_acc,
    'saved_at': datetime.now().isoformat()
}

torch.save(checkpoint, checkpoint_path)
print(f"Model saved to {checkpoint_path}")

In [ ]:
# Save training history as JSON
history_path = CHECKPOINT_DIR / 'training_history.json'

with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print(f"Training history saved to {history_path}")

In [ ]:
# Demonstrate loading the model
print("Demonstrating model loading...")

# Create a new model instance
loaded_model = BGRUModel(
    input_dim=5,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3
).to(device)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
loaded_model.load_state_dict(checkpoint['model_state_dict'])

print(f"Model loaded from {checkpoint_path}")
print(f"  Saved at: {checkpoint['saved_at']}")
print(f"  Best Val Acc: {checkpoint['best_val_acc']:.4f}")

# Verify loaded model works
loaded_model.eval()
with torch.no_grad():
    sample_output = loaded_model(X_test_tensor[:8])
print(f"\nLoaded model test output: {sample_output.flatten()[:4].tolist()}")
print("Model loading verified!")

In [ ]:
# Summary
print("\n" + "=" * 60)
print("BGRU BASELINE MODEL - SUMMARY")
print("=" * 60)
print(f"\nModel Architecture:")
print(f"  - Input: [batch, {SEQUENCE_LENGTH}, 5] (OHLCV)")
print(f"  - BGRU Layer 1: 128 units, bidirectional")
print(f"  - BGRU Layer 2: 64 units, bidirectional")
print(f"  - Dense: 32 units, ReLU")
print(f"  - Output: 1 unit, Sigmoid")
print(f"  - Total Parameters: {total_params:,}")

print(f"\nTraining:")
print(f"  - Epochs trained: {len(history['train_loss'])}")
print(f"  - Best Val Accuracy: {best_val_acc:.4f}")
print(f"  - Final Test Accuracy: {test_acc:.4f}")

print(f"\nFiles Saved:")
print(f"  - Model: {checkpoint_path}")
print(f"  - History: {history_path}")
print(f"  - Plots: {CHECKPOINT_DIR / 'training_curves.png'}")

print("\n" + "=" * 60)
print("Notebook complete!")
print("=" * 60)